# R03 Lever Ablations - H14 seeding, H15 decomposition, H16 head+tail

**Author**: Knowledge Graph Foundry autonomous build

Each lever toggled independently on the densified R01 graph with propositions on and the strong reader, abstention off (H17 already adjudicated). H15 measured on the 8 comparison probes only (its target population); H14 and H16 on all 24 answerable probes.

In [ ]:
# imports
import datetime
import json
import re
from pathlib import Path

import yaml

from knowledge_graph_foundry import Foundry, load_settings

In [ ]:
# probes + scoring (same rules as probe_eval.ipynb)
probes = yaml.safe_load(Path('../tests/probes/cpap-probe-set.yml').read_text())
answerable = [p for p in probes if p['category'] != 'unanswerable']
comparisons = [p for p in probes if p['category'] == 'comparison']

def _norm(s):
    return re.sub(r'\s+', ' ', s.casefold())

def evidence_recall(gold, context):
    ctx = _norm(context)
    return sum(1 for g in gold if _norm(g) in ctx) / len(gold) if gold else None

def value_tokens(gold_answer):
    return re.findall(r'[\w.\-/]*\d[\w.\-/]*', gold_answer)

def answer_correct(probe, answer):
    ans = _norm(answer)
    tokens = value_tokens(probe['gold_answer'])
    if tokens:
        hit = sum(1 for t in tokens if _norm(t) in ans)
        return hit >= max(1, len(tokens) // 2 + (len(tokens) % 2))
    return _norm(probe['gold_answer']) in ans

def base_settings():
    s = load_settings(Path('../config.yml') if Path('../config.yml').exists() else None)
    s.graphrag.propositions_enabled = True
    s.graphrag.abstention_enabled = False
    return s

def run(settings, probe_list, label, with_recall=True):
    rows = []
    with Foundry(settings) as f:
        for p in probe_list:
            rec = None
            if with_recall:
                lines, _, _ = f._retrieve_local(p['question'])
                rec = evidence_recall(p['gold_evidence'], '\n'.join(lines))
            ans = f.query(p['question'])['answer']
            rows.append({'id': p['id'], 'recall': rec,
                         'correct': answer_correct(p, ans), 'answer': ans})
            print(p['id'], rows[-1]['recall'], rows[-1]['correct'])
    recs = [r['recall'] for r in rows if r['recall'] is not None]
    return {'label': label, 'rows': rows,
            'recall': sum(recs) / len(recs) if recs else None,
            'accuracy': sum(r['correct'] for r in rows) / len(rows)}

In [ ]:
# H14: proposition seeding off vs on (context identical, seeds differ)
s = base_settings(); s.graphrag.proposition_seeding = False
h14_off = run(s, answerable, 'H14_seeding_off')
s = base_settings(); s.graphrag.proposition_seeding = True
h14_on = run(s, answerable, 'H14_seeding_on')
print('H14:', h14_off['recall'], '->', h14_on['recall'],
      '| acc', h14_off['accuracy'], '->', h14_on['accuracy'])

In [ ]:
# H15: comparison decomposition off vs on (comparison probes only)
s = base_settings(); s.graphrag.decompose_comparisons = False
h15_off = run(s, comparisons, 'H15_decompose_off')
s = base_settings(); s.graphrag.decompose_comparisons = True
h15_on = run(s, comparisons, 'H15_decompose_on')
print('H15 comparisons:', h15_off['accuracy'], '->', h15_on['accuracy'],
      '| recall', h15_off['recall'], '->', h15_on['recall'])

In [ ]:
# H16: head+tail interleave off vs on (all answerable, accuracy focus)
s = base_settings(); s.graphrag.context_head_tail = False
h16_off = run(s, answerable, 'H16_headtail_off', with_recall=False)
s = base_settings(); s.graphrag.context_head_tail = True
h16_on = run(s, answerable, 'H16_headtail_on', with_recall=False)
print('H16:', h16_off['accuracy'], '->', h16_on['accuracy'])

In [ ]:
# persist
results = {r['label']: r for r in
           (h14_off, h14_on, h15_off, h15_on, h16_off, h16_on)}
for r in results.values():
    print(f"{r['label']:<20} recall={r['recall']} acc={r['accuracy']:.3f}")
stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d-%H%M%S')
out = Path('../reports') / f'probe-eval-r03-{stamp}.json'
out.write_text(json.dumps(results, indent=2, default=str))
print('saved', out)